# 🚀 GridForecast — 05: Boosting (XGBoost + LightGBM)

**Цел:** Production-grade boosting модел за 1-step recursive forecast на `consumption_mw`.

**Защо boosting:**
- SARIMA v2: MAPE 8.07% (Ljung-Box FAIL → residuals имат автокорелация)
- Seasonal Naive: MAPE ~9.20%
- Очаквано за boosting: MAPE ~2-3%

**Anti-leakage правила:**
- Lag features: `.shift(lag)`
- Rolling features: `.shift(1).rolling(w)` (shift **преди** rolling)
- `drift_factor` е изключен от features (ground truth → leakage)
- Scaler: `fit` само на train, `transform` на test

**Два walk-forward експеримента:**
- **8a. Naive split (80/20)** — стандартен, но при `sudden` drift-ът попада в train → "измамен" резултат
- **8b. Drift-aware split** — split на day 365 за `sudden`/`gradual`, за да хванем drift-а в test → "честен" резултат

**Export за API-то (Фаза B):** `gridforecast_xgb.joblib` + `scaler.joblib` + `feature_columns.json`

## 1️⃣ Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from gridforecast.config import (
    SYNTHETIC_DIR,
    MODELS_DIR,
    FIGURES_DIR,
    TRAIN_RATIO,
    RANDOM_SEED,
    LAG_FEATURES,
    ROLLING_WINDOWS,
    BOOSTING_PARAMS,
    FORECAST_HORIZON_HOURS,
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titleweight"] = "bold"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(RANDOM_SEED)

print("✅ Imports OK")
print(f"   LAG_FEATURES:    {LAG_FEATURES}")
print(f"   ROLLING_WINDOWS: {ROLLING_WINDOWS}")
print(f"   HORIZON:         {FORECAST_HORIZON_HOURS}h")

✅ Imports OK
   LAG_FEATURES:    [1, 2, 3, 24, 48, 168]
   ROLLING_WINDOWS: [24, 168]
   HORIZON:         24h


## 2️⃣ Load + feature engineering

**Anti-leakage:** lag = `.shift(lag)`; rolling = `.shift(1).rolling(w)`; `drift_factor` изключен.

In [2]:
df = pd.read_parquet(SYNTHETIC_DIR / "consumption_stable.parquet")
df.index = pd.DatetimeIndex(df.index)
print(f"Raw: {df.shape} | {df.index.min()} → {df.index.max()}")

# ✅ Single source of truth — идва от gridforecast.features
from gridforecast.features import (
    create_features,
    save_feature_columns,
    FEATURE_COLUMNS_FILE,
)

df_feat = create_features(df).dropna()

# ⚠️ EXCLUDE: 7 колони
#    - consumption_mw: target
#    - drift_factor: ground truth → leakage
#    - hour/day_of_week/day_of_year/month: raw calendar (дублират *_sin/*_cos)
#    - dow: дублира day_of_week (създадена от features.py)
EXCLUDE = {
    "consumption_mw",
    "drift_factor",
    "hour", "day_of_week", "day_of_year", "month",
    "dow",
}
feature_cols = [c for c in df_feat.columns if c not in EXCLUDE]

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"After dropna: {df_feat.shape}")

# 💾 Записваме feature_columns.json (23 features, без leakage)
save_feature_columns(feature_cols)
print(f"✅ Saved → {FEATURE_COLUMNS_FILE}")

df_feat.head()

Raw: (17520, 9) | 2023-01-01 00:00:00 → 2024-12-30 23:00:00
Features (23): ['temperature_c', 'is_holiday', 'is_weekend', 'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'lag_48h', 'lag_168h', 'rolling_mean_24h', 'rolling_std_24h', 'rolling_min_24h', 'rolling_max_24h', 'rolling_mean_168h', 'rolling_std_168h', 'rolling_min_168h', 'rolling_max_168h', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
After dropna: (17352, 30)
✅ Saved → D:\CUDA\Python\ClassicalML\GridForecast\models\feature_columns.json


,consumption_mw,temperature_c,is_holiday,is_weekend,hour,day_of_week,day_of_year,month,drift_factor,lag_1h,...,rolling_std_168h,rolling_min_168h,rolling_max_168h,dow,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos
timestamp,,,,,,,,,,,,,,,,,,,,,
2023-01-08 00:00:00,1327.954487,-1.786639,False,True,0,6,8,1,1.0,1365.029683,...,279.236031,777.126041,1900.80501,6,0.000000,1.000000,-0.781831,0.62349,0.5,0.866025
2023-01-08 01:00:00,1345.005311,-2.349481,False,True,1,6,8,1,1.0,1327.954487,...,277.418984,777.126041,1900.80501,6,0.258819,0.965926,-0.781831,0.62349,0.5,0.866025
2023-01-08 02:00:00,1407.320124,-4.077033,False,True,2,6,8,1,1.0,1345.005311,...,276.000718,777.126041,1900.80501,6,0.500000,0.866025,-0.781831,0.62349,0.5,0.866025
2023-01-08 03:00:00,1308.147844,-2.045684,False,True,3,6,8,1,1.0,1407.320124,...,274.098795,777.126041,1900.80501,6,0.707107,0.707107,-0.781831,0.62349,0.5,0.866025
2023-01-08 04:00:00,1360.604959,-3.677425,False,True,4,6,8,1,1.0,1308.147844,...,272.229028,777.126041,1900.80501,6,0.866025,0.500000,-0.781831,0.62349,0.5,0.866025


## 3️⃣ Train/test split (chronological 80/20)

In [3]:
split_idx = int(TRAIN_RATIO * len(df_feat))
train = df_feat.iloc[:split_idx]
test  = df_feat.iloc[split_idx:]

print(f"Train: {len(train):,} | {train.index.min()} → {train.index.max()}")
print(f"Test:  {len(test):,} | {test.index.min()} → {test.index.max()}")

scaler = StandardScaler()
X_train = scaler.fit_transform(train[feature_cols])
X_test  = scaler.transform(test[feature_cols])
y_train = train["consumption_mw"].values
y_test  = test["consumption_mw"].values

val_size = int(0.15 * len(X_train))
X_tr, X_val = X_train[:-val_size], X_train[-val_size:]
y_tr, y_val = y_train[:-val_size], y_train[-val_size:]

print(f"\nTrain (for fit):   {len(X_tr):,}")
print(f"Val (early stop):  {len(X_val):,}")
print(f"Test:              {len(X_test):,}")

Train: 13,881 | 2023-01-08 00:00:00 → 2024-08-08 08:00:00
Test:  3,471 | 2024-08-08 09:00:00 → 2024-12-30 23:00:00

Train (for fit):   11,799
Val (early stop):  2,082
Test:              3,471


## 4️⃣ LightGBM

In [4]:
def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


t0 = time.time()
lgb_model = lgb.LGBMRegressor(**BOOSTING_PARAMS)
lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
lgb_time = time.time() - t0

y_pred_lgb = lgb_model.predict(X_test)

print(f"⏱️  LightGBM trained in {lgb_time:.1f}s")
print(f"   best_iteration: {getattr(lgb_model, 'best_iteration_', getattr(lgb_model, 'best_iteration', 'n/a'))}")
print(f"   MAE:  {mean_absolute_error(y_test, y_pred_lgb):.2f} MW")
print(f"   RMSE: {rmse(y_test, y_pred_lgb):.2f} MW")
print(f"   MAPE: {mape(y_test, y_pred_lgb):.2f}%")

⏱️  LightGBM trained in 1.6s
   best_iteration: 1334
   MAE:  29.35 MW
   RMSE: 37.26 MW
   MAPE: 2.97%


## 5️⃣ XGBoost

In [5]:
xgb_params = {
    "n_estimators": 2000,
    "learning_rate": 0.03,
    "max_depth": 6,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "tree_method": "hist",
    "early_stopping_rounds": 50,
    "eval_metric": "mape",
}

t0 = time.time()
xgb_model = xgb.XGBRegressor(**xgb_params)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
xgb_time = time.time() - t0

y_pred_xgb = xgb_model.predict(X_test)

print(f"⏱️  XGBoost trained in {xgb_time:.1f}s")
print(f"   best_iteration: {xgb_model.best_iteration}")
print(f"   MAE:  {mean_absolute_error(y_test, y_pred_xgb):.2f} MW")
print(f"   RMSE: {rmse(y_test, y_pred_xgb):.2f} MW")
print(f"   MAPE: {mape(y_test, y_pred_xgb):.2f}%")

⏱️  XGBoost trained in 1.7s
   best_iteration: 1762
   MAE:  29.26 MW
   RMSE: 37.12 MW
   MAPE: 2.96%


## 6️⃣ Сравнение на модели

In [6]:
comparison = pd.DataFrame([
    {"model": "LightGBM", "MAE": mean_absolute_error(y_test, y_pred_lgb),
     "RMSE": rmse(y_test, y_pred_lgb), "MAPE": mape(y_test, y_pred_lgb), "time_s": lgb_time},
    {"model": "XGBoost", "MAE": mean_absolute_error(y_test, y_pred_xgb),
     "RMSE": rmse(y_test, y_pred_xgb), "MAPE": mape(y_test, y_pred_xgb), "time_s": xgb_time},
]).sort_values("MAPE").reset_index(drop=True)

print(comparison.to_string(index=False))

print("\n— Baselines (от 02_baselines / 03_arima):")
print("  Seasonal Naive: MAPE ~9.20%")
print("  SARIMA v2:      MAPE ~8.07%")

   model       MAE      RMSE     MAPE   time_s
 XGBoost 29.258233 37.124942 2.963968 1.669949
LightGBM 29.346340 37.256754 2.970520 1.629673

— Baselines (от 02_baselines / 03_arima):
  Seasonal Naive: MAPE ~9.20%
  SARIMA v2:      MAPE ~8.07%


## 7️⃣ Feature importance (leakage check)

In [7]:
best_name = comparison.iloc[0]["model"]
best_model = lgb_model if best_name == "LightGBM" else xgb_model
print(f"Best: {best_name}")

if best_name == "LightGBM":
    imp = pd.DataFrame({
        "feature": feature_cols,
        "importance": best_model.booster_.feature_importance(importance_type="gain"),
    })
else:
    imp = pd.DataFrame({
        "feature": feature_cols,
        "importance": best_model.feature_importances_,
    })

imp["pct"] = imp["importance"] / imp["importance"].sum() * 100
imp = imp.sort_values("importance", ascending=False).reset_index(drop=True)

print("\nTop-10 features:")
print(imp.head(10).to_string(index=False))

if imp.iloc[0]["pct"] > 50:
    print(f"\n🚨 LEAKAGE ALERT: '{imp.iloc[0]['feature']}' = {imp.iloc[0]['pct']:.1f}%")
else:
    print(f"\n✅ No single feature >50% (top: {imp.iloc[0]['feature']} = {imp.iloc[0]['pct']:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 7))
top = imp.head(20)
ax.barh(top["feature"][::-1], top["pct"][::-1], color="#3498db", edgecolor="black")
ax.set_xlabel("% of total gain")
ax.set_title(f"Top-20 feature importance ({best_name})")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05a_feature_importance.png", bbox_inches="tight")
plt.show()

Best: XGBoost

Top-10 features:
      feature  importance       pct
       lag_1h    0.381796 38.179596
     lag_168h    0.163666 16.366581
   is_holiday    0.101441 10.144093
      lag_24h    0.091270  9.126958
   is_weekend    0.043129  4.312934
       lag_2h    0.033929  3.392918
      dow_sin    0.028444  2.844362
      dow_cos    0.023989  2.398884
temperature_c    0.023732  2.373219
     hour_sin    0.020213  2.021265

✅ No single feature >50% (top: lag_1h = 38.2%)


## 8a. Walk-forward — naive 80/20 split

**Внимание:** при `sudden` drift-ът (day 365) попада в train (test е от day 584) → MAPE е "измамен". Виж 8b за "честен" резултат.

In [8]:
DRIFT_TYPES = ["stable", "sudden", "gradual", "recurring"]
scenario_results = []
scenario_preds = {}

for drift in DRIFT_TYPES:
    d = pd.read_parquet(SYNTHETIC_DIR / f"consumption_{drift}.parquet")
    d.index = pd.DatetimeIndex(d.index)
    d_feat = create_features(d).dropna()

    split = int(TRAIN_RATIO * len(d_feat))
    tr, te = d_feat.iloc[:split], d_feat.iloc[split:]

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(tr[feature_cols])
    X_te_s = sc.transform(te[feature_cols])
    y_tr_s = tr["consumption_mw"].values
    y_te_s = te["consumption_mw"].values

    val = int(0.15 * len(X_tr_s))
    m = lgb.LGBMRegressor(**BOOSTING_PARAMS)
    m.fit(X_tr_s[:-val], y_tr_s[:-val],
          eval_set=[(X_tr_s[-val:], y_tr_s[-val:])],
          callbacks=[lgb.early_stopping(50, verbose=False)])

    yp = m.predict(X_te_s)
    scenario_results.append({"scenario": drift, "MAE": mean_absolute_error(y_te_s, yp),
                              "RMSE": rmse(y_te_s, yp), "MAPE": mape(y_te_s, yp)})
    scenario_preds[drift] = (te.index, y_te_s, yp)

scenario_df = pd.DataFrame(scenario_results)
print("Naive 80/20 split:")
print(scenario_df.to_string(index=False))

Naive 80/20 split:
 scenario       MAE      RMSE     MAPE
   stable 29.346340 37.256754 2.970520
   sudden 28.836802 36.792766 2.920660
  gradual 47.391477 61.115923 3.746877
recurring 29.794466 38.051967 3.136465


## 8b. Walk-forward — drift-aware split

За `sudden` и `gradual`: split на **day 365** (преди drift-а), за да хванем drift-а в test.

- `sudden`: drift на day 365, recovery 90 дни → test day 365–730 хваща recovery
- `gradual`: drift day 180–730 → test day 365–730 хваща късната drift фаза
- `stable` / `recurring`: остават с 80/20 (няма drift за хващане)

In [9]:
DRIFT_DAY = 365
drift_aware_results = []
drift_aware_preds = {}

for drift in DRIFT_TYPES:
    d = pd.read_parquet(SYNTHETIC_DIR / f"consumption_{drift}.parquet")
    d.index = pd.DatetimeIndex(d.index)
    d_feat = create_features(d).dropna()

    # Drift-aware split: day 365 за sudden/gradual, 80/20 за stable/recurring
    if drift in ("sudden", "gradual"):
        cutoff = d_feat.index[0] + pd.Timedelta(days=DRIFT_DAY)
        tr = d_feat[d_feat.index < cutoff]
        te = d_feat[d_feat.index >= cutoff]
    else:
        split = int(TRAIN_RATIO * len(d_feat))
        tr, te = d_feat.iloc[:split], d_feat.iloc[split:]

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(tr[feature_cols])
    X_te_s = sc.transform(te[feature_cols])
    y_tr_s = tr["consumption_mw"].values
    y_te_s = te["consumption_mw"].values

    val = int(0.15 * len(X_tr_s))
    m = lgb.LGBMRegressor(**BOOSTING_PARAMS)
    m.fit(X_tr_s[:-val], y_tr_s[:-val],
          eval_set=[(X_tr_s[-val:], y_tr_s[-val:])],
          callbacks=[lgb.early_stopping(50, verbose=False)])

    yp = m.predict(X_te_s)
    drift_aware_results.append({"scenario": drift,
                                 "n_train": len(tr), "n_test": len(te),
                                 "MAE": mean_absolute_error(y_te_s, yp),
                                 "RMSE": rmse(y_te_s, yp),
                                 "MAPE": mape(y_te_s, yp)})
    drift_aware_preds[drift] = (te.index, y_te_s, yp)

drift_aware_df = pd.DataFrame(drift_aware_results)
print("Drift-aware split (day 365 за sudden/gradual):")
print(drift_aware_df.to_string(index=False))

Drift-aware split (day 365 за sudden/gradual):
 scenario  n_train  n_test       MAE      RMSE     MAPE
   stable    13881    3471 29.346340 37.256754 2.970520
   sudden     8760    8592 40.696992 53.114611 3.810981
  gradual     8760    8592 64.948625 84.995382 4.715485
recurring    13881    3471 29.794466 38.051967 3.136465


### 8c. Сравнение: naive vs drift-aware

За `sudden`/`gradual` очакваме drift-aware MAPE да е **значително по-висок** от naive → това е "честният" резултат и мотивацията за drift detection.

In [10]:
compare = scenario_df[["scenario", "MAPE"]].rename(columns={"MAPE": "naive_mape"})
compare = compare.merge(
    drift_aware_df[["scenario", "MAPE"]].rename(columns={"MAPE": "drift_aware_mape"}),
    on="scenario", how="left",
)
compare["delta"] = compare["drift_aware_mape"] - compare["naive_mape"]
print(compare.to_string(index=False))

print("\n💡 Интерпретация:")
print("  - stable/recurring: почти без разлика (drift-ът е в train и в двата split-а)")
print("  - sudden/gradual: drift-aware MAPE е по-висок → 'честен' резултат → мотивация за drift detection")

# MAPE по месеци в test периода (за sudden/gradual, drift-aware split)
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)
for ax, drift in zip(axes, ["sudden", "gradual"]):
    idx, yt, yp = drift_aware_preds[drift]
    err = pd.DataFrame({"y": yt, "p": yp}, index=idx)
    err["mape"] = np.abs((err["y"] - err["p"]) / err["y"]) * 100
    monthly = err["mape"].resample("ME").mean()
    ax.plot(monthly.index, monthly.values, marker="o", color="#e74c3c", lw=2)
    ax.set_title(f"{drift.upper()} — MAPE по месеци (drift-aware split)", loc="left")
    ax.set_ylabel("MAPE (%)")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05c_drift_aware_monthly.png", bbox_inches="tight")
plt.show()

 scenario  naive_mape  drift_aware_mape    delta
   stable    2.970520          2.970520 0.000000
   sudden    2.920660          3.810981 0.890321
  gradual    3.746877          4.715485 0.968609
recurring    3.136465          3.136465 0.000000

💡 Интерпретация:
  - stable/recurring: почти без разлика (drift-ът е в train и в двата split-а)
  - sudden/gradual: drift-aware MAPE е по-висок → 'честен' резултат → мотивация за drift detection


## 9️⃣ Визуализации — predictions vs actual по сценарий (naive split)

In [11]:
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=False)

for ax, drift in zip(axes, DRIFT_TYPES):
    idx, yt, yp = scenario_preds[drift]
    n_show = min(7 * 24, len(idx))
    ax.plot(idx[:n_show], yt[:n_show], label="Actual", color="#2c3e50", lw=1.5)
    ax.plot(idx[:n_show], yp[:n_show], label="Predicted", color="#e74c3c", lw=1.5, ls="--")
    m = scenario_df.loc[scenario_df.scenario == drift, "MAPE"].values[0]
    ax.set_title(f"{drift.upper()} — MAPE = {m:.2f}%  (first 7 days of test)", loc="left")
    ax.set_ylabel("MW")
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)

plt.suptitle("Boosting forecast — 4 drift scenarios (naive 80/20 split)",
             fontsize=14, fontweight="bold", y=1.00)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05b_walkforward_4scenarios.png", bbox_inches="tight")
plt.show()

## 🔟 Joblib export (production artifact)

Записваме в `MODELS_DIR`:
- `gridforecast_xgb.joblib` — best моделът (naive split, за да сервира на "нормални" данни)
- `scaler.joblib` — StandardScaler
- `feature_columns.json` — feature list + drift-aware MAPE

In [12]:
import joblib
from gridforecast.features import save_feature_columns, FEATURE_COLUMNS_FILE

model_path = MODELS_DIR / "gridforecast_xgb.joblib"
scaler_path = MODELS_DIR / "scaler.joblib"
metadata_path = MODELS_DIR / "model_metadata.json"

# 1. Записваме модела и scaler-а
joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)

# 2. Записваме features в ЕДИНСТВЕНИЯ правилен формат (от features.py)
save_feature_columns(feature_cols)

# 3. Метаданни (отделно, за да не се бърка с features)
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump({
        "target": "consumption_mw",
        "horizon_hours": FORECAST_HORIZON_HOURS,
        "model_type": best_name,
        "n_features": len(feature_cols),
        "cv_mape_naive": {r["scenario"]: r["MAPE"] for r in scenario_results},
        "cv_mape_drift_aware": {r["scenario"]: r["MAPE"] for r in drift_aware_results},
    }, f, indent=2, ensure_ascii=False)

print(f"💾 {model_path}")
print(f"💾 {scaler_path}")
print(f"💾 {FEATURE_COLUMNS_FILE}")
print(f"💾 {metadata_path}")

# 4. Проверка — че всичко се чете обратно коректно
loaded_model = joblib.load(model_path)
loaded_scaler = joblib.load(scaler_path)
from gridforecast.features import load_feature_columns
loaded_features = load_feature_columns()
with open(metadata_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

assert len(loaded_features) == len(feature_cols), f"Feature mismatch: {len(loaded_features)} != {len(feature_cols)}"
assert loaded_features == feature_cols, "Feature order mismatch!"
assert meta["n_features"] == len(feature_cols)

print(f"\n✅ Verified:")
print(f"   model:    {type(loaded_model).__name__}")
print(f"   scaler:   {type(loaded_scaler).__name__}")
print(f"   features: {len(loaded_features)} (order matches)")
print(f"   metadata: {metadata_path.name}")

💾 D:\CUDA\Python\ClassicalML\GridForecast\models\gridforecast_xgb.joblib
💾 D:\CUDA\Python\ClassicalML\GridForecast\models\scaler.joblib
💾 D:\CUDA\Python\ClassicalML\GridForecast\models\feature_columns.json
💾 D:\CUDA\Python\ClassicalML\GridForecast\models\model_metadata.json

✅ Verified:
   model:    XGBRegressor
   scaler:   StandardScaler
   features: 23 (order matches)
   metadata: model_metadata.json


## 1️⃣1️⃣ Заключение

In [13]:
print("=" * 70)
print("ЗАКЛЮЧЕНИЕ (05_boosting.ipynb)")
print("=" * 70)
print()
print(f"Best model: {best_name}  (stable MAPE = {comparison.iloc[0]['MAPE']:.2f}%)")
print()
print("Сравнение с предишни модели (stable):")
print(f"  Seasonal Naive:  ~9.20%")
print(f"  SARIMA v2:       ~8.07%")
print(f"  {best_name:16s} {comparison.iloc[0]['MAPE']:>5.2f}%  ←{' win' if comparison.iloc[0]['MAPE'] < 8.07 else ''}")
print()
print("Walk-forward — naive 80/20 split:")
for _, row in scenario_df.iterrows():
    print(f"  {row['scenario']:10s} MAPE = {row['MAPE']:>5.2f}%")
print()
print("Walk-forward — drift-aware split (day 365):")
for _, row in drift_aware_df.iterrows():
    print(f"  {row['scenario']:10s} MAPE = {row['MAPE']:>5.2f}%  (n_test={row['n_test']:,})")
print()
print("Ключов извод:")
print("  - sudden: naive split крие drift-а в train → 'измамен' MAPE 2.92%")
print("  - drift-aware split показва реалния ефект → мотивация за drift detection")
print()
print("Exported for API (Фаза B):")
print(f"  {model_path.name}")
print(f"  {scaler_path.name}")
print(f"  {FEATURE_COLUMNS_FILE.name}")   # ← поправено
print(f"  {metadata_path.name}")          # ← ново
print()
print("Следва:")
print("  - Фаза B: production scaffold (FastAPI + Streamlit + Docker + Prometheus + Grafana)")
print("  - 07_drift.ipynb: ADWIN/CUSUM на residuals, използвайки drift-aware резултатите тук")
print("=" * 70)

ЗАКЛЮЧЕНИЕ (05_boosting.ipynb)

Best model: XGBoost  (stable MAPE = 2.96%)

Сравнение с предишни модели (stable):
  Seasonal Naive:  ~9.20%
  SARIMA v2:       ~8.07%
  XGBoost           2.96%  ← win

Walk-forward — naive 80/20 split:
  stable     MAPE =  2.97%
  sudden     MAPE =  2.92%
  gradual    MAPE =  3.75%
  recurring  MAPE =  3.14%

Walk-forward — drift-aware split (day 365):
  stable     MAPE =  2.97%  (n_test=3,471)
  sudden     MAPE =  3.81%  (n_test=8,592)
  gradual    MAPE =  4.72%  (n_test=8,592)
  recurring  MAPE =  3.14%  (n_test=3,471)

Ключов извод:
  - sudden: naive split крие drift-а в train → 'измамен' MAPE 2.92%
  - drift-aware split показва реалния ефект → мотивация за drift detection

Exported for API (Фаза B):
  gridforecast_xgb.joblib
  scaler.joblib
  feature_columns.json
  model_metadata.json

Следва:
  - Фаза B: production scaffold (FastAPI + Streamlit + Docker + Prometheus + Grafana)
  - 07_drift.ipynb: ADWIN/CUSUM на residuals, използвайки drift-aware рез